In [16]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
from google.colab import userdata

folderPath = userdata.get("reinforcementLearningFolderPath")

In [18]:
roms_path = f"{folderPath}/Project/roms"

!mkdir -p roms
!cp -r "$roms_path" ./

In [19]:
!pip install -q stable-retro coloredlogs

In [20]:
!python -m retro.import ./roms

Importing SuperMarioWorld-Snes-v0
Imported 1 games


In [21]:
import coloredlogs
import logging
from datetime import datetime
from logging import Logger, FileHandler, Formatter, LoggerAdapter

def get_logger(run_name: str, level: str):
    logger = logging.getLogger(run_name)
    logger.setLevel(level)
    coloredlogs.install(level=level, logger=logger, fmt="%(asctime)s [%(levelname)s] %(message)s", isatty=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file_name = f"{run_name}_{timestamp}.log"
    file_handler = FileHandler(log_file_name)
    file_handler.setLevel(level)
    formatter = Formatter("%(asctime)s [%(levelname)s] %(message)s")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    logger.info(f"Session log for run {run_name} with level [{level}] initialized at: {log_file_name}")
    return logger

class DummyLogger(Logger):
    def __init__(self):
        super().__init__("dummy")
    def debug(self, msg, *args, **kwargs):
        pass
    def info(self, msg, *args, **kwargs):
        pass
    def warning(self, msg, *args, **kwargs):
        pass
    def error(self, msg, *args, **kwargs):
        pass
    def critical(self, msg, *args, **kwargs):
        pass
    def exception(self, msg, *args, **kwargs):
        pass

# Initial Testing

In [22]:
import json

def update_emulator_setup(setup: dict):
    path = '/usr/local/lib/python3.12/dist-packages/stable_retro/data/stable/SuperMarioWorld-Snes-v0/data.json'
    with open(path, 'w') as f:
        json.dump(setup, f, indent=2)

In [23]:
emulator_setup = {
  "info": {
    "score":        {"address": 8261428, "type": "<u4"},
    "lives":        {"address": 8261054, "type": "|i1"},
    "x":            {"address": 0x94, "type": "<u2"},  # Mario X position (16-bit)
    "y":            {"address": 0x96, "type": "<u2"},  # Mario Y position (16-bit)
    "cam_x":        {"address": 0x1A,   "type": "<u2"}, # $7E:001A (Camera Scroll X)
    "cam_y":        {"address": 0x1C,   "type": "<u2"}, # $7E:001C (Camera Scroll Y)
    "mario_powerup": {"address": 0x0019, "type": "|u1"}, # 0=Small, 1=Big, 2=Cape, 3=Fire
    "mario_climbing": {"address": 0x0074, "type": "|u1"}, # Non-zero if on a vine/ladder
    "mario_y_screen": {"address": 0x00D3, "type": "|u1"}, # Vertical screen Mario is on
    "mario_priority": {"address": 0x13F9, "type": "|u1"}, # Layering priority for Mario
  }
}

for i in range(12):
    emulator_setup["info"][f"sprite_status_{i}"] = {"address": 0x14C8 + i, "type": "|u1"} # 0 = Empty, 8 = Active
    emulator_setup["info"][f"sprite_type_{i}"]   = {"address": 0x009E + i, "type": "|u1"} # What is it? (Goomba, Koopa, etc.)
    emulator_setup["info"][f"sprite_x_low_{i}"]  = {"address": 0x00E4 + i, "type": "|u1"}
    emulator_setup["info"][f"sprite_x_high_{i}"] = {"address": 0x14E0 + i, "type": "|u1"}
    emulator_setup["info"][f"sprite_y_low_{i}"]  = {"address": 0x00D8 + i, "type": "|u1"}
    emulator_setup["info"][f"sprite_y_high_{i}"] = {"address": 0x14D4 + i, "type": "|u1"}
    emulator_setup["info"][f"sprite_priority_{i}"] = {"address": 0x15F6 + i, "type": "|u1"}

for i in range(10):
    emulator_setup["info"][f"ext_sprite_type_{i}"]   = {"address": 0x170B + i, "type": "|u1"}
    emulator_setup["info"][f"ext_sprite_x_low_{i}"]  = {"address": 0x171F + i, "type": "|u1"}
    emulator_setup["info"][f"ext_sprite_x_high_{i}"] = {"address": 0x1733 + i, "type": "|u1"}
    emulator_setup["info"][f"ext_sprite_y_low_{i}"]  = {"address": 0x1715 + i, "type": "|u1"}
    emulator_setup["info"][f"ext_sprite_y_high_{i}"] = {"address": 0x1729 + i, "type": "|u1"}

WRAM_BASE = 8257536
TILE_LAYER_1_LOW = 0xC800
OFFSET_TILE_LAYER_1_LOW = TILE_LAYER_1_LOW + WRAM_BASE

for i in range(512):
    emulator_setup["info"][f"tile_layer1_low_{i}"] = {"address": OFFSET_TILE_LAYER_1_LOW + i, "type": "|u1"}

update_emulator_setup(emulator_setup)

In [24]:
import enum
from typing import TypedDict, Optional

class TileType(enum.Enum):
    EMPTY = 0
    MARIO = 1
    SPRITE = 2
    EXTENDED_SPRITE = 3
    TERRAIN = 4
    BLOCK = 5

class Tile(TypedDict):
    type: TileType
    id: int

def tile_absolute_id(tile: Tile) -> int:
    if tile["type"] == TileType.EMPTY:
        return 0
    if tile["type"] == TileType.MARIO:
        return 1
    offset_level = 0
    offset_multiplier = 256
    if tile["type"] == TileType.SPRITE:
        offset_level = 1
    if tile["type"] == TileType.EXTENDED_SPRITE:
        offset_level = 2
    if tile["type"] == TileType.TERRAIN:
        offset_level = 3
    if tile["type"] == TileType.BLOCK:
        offset_level = 4
    offset = offset_level * offset_multiplier
    return int(tile["id"]) + offset

class LayeredTile():
    def __init__(self, type: TileType, id: int, layer: int):
        self.type = type
        self.id = id
        self.layer = layer

    def get_tile(self) -> Tile:
        return Tile(type=self.type, id=self.id)

class UngriddedTile(LayeredTile):
    def __init__(self, type: TileType, id: int, layer: int, x: int, y: int):
        super().__init__(type, id, layer)
        self.x = x
        self.y = y

class TileStack():
    def __init__(self):
        self.stack: list[LayeredTile] = []

    def add(self, tile: LayeredTile):
        self.stack.append(tile)

    def resolve(self) -> Tile:
        if len(self.stack) == 0:
            return Tile(type=TileType.EMPTY, id=0)
        front_tile = self.stack[0]
        for tile in self.stack:
            if tile.layer < front_tile.layer:
                front_tile = tile
        return front_tile.get_tile()

class DenseTileMatrix():
    def __init__(self, rows: int = 14, columns: int = 16):
        self.rows = rows
        self.columns = columns
        self.matrix: list[list[TileStack]] = []
        for row in range(rows):
            row_stack = []
            for col in range(columns):
                row_stack.append(TileStack())
            self.matrix.append(row_stack)

    def add(self, tile: Optional[UngriddedTile]):
        if tile is None:
            return
        if not self._are_grid_coords_in_cam_view(tile.x, tile.y):
            return
        self.matrix[tile.y][tile.x].add(tile)

    def _are_grid_coords_in_cam_view(self, grid_x: int, grid_y: int) -> bool:
        return 0 <= grid_x < self.columns and 0 <= grid_y < self.rows

    def resolve(self):
        resolved_matrix = []
        for row in self.matrix:
            resolved_row = []
            for col in row:
                resolved_row.append(col.resolve())
            resolved_matrix.append(resolved_row)
        return resolved_matrix

    def resolveSimplified(self):
        resolved_matrix = []
        for row in self.matrix:
            resolved_row = []
            for col in row:
                resolved_row.append(tile_absolute_id(col.resolve()))
            resolved_matrix.append(resolved_row)
        return np.array(resolved_matrix)

In [25]:
from typing import Optional

import numpy as np

class WorldParser:
    def __init__(self, env, logger: Optional[Logger] = None):
        self.logger: Logger = logger if logger is not None else DummyLogger()
        self.env = env
        self.SCREEN_ROWS = 14
        self.SCREEN_COLUMNS = 16
        self.TILE_SIZE = 16

    def get_screen_matrix(self, info) -> np.matrix:
        screen_matrix = self._build_dense_screen_matrix(info)
        return screen_matrix.resolve()

    def get_screen_matrix_simplified(self, info) -> np.matrix:
        screen_matrix = self._build_dense_screen_matrix(info)
        return screen_matrix.resolveSimplified()

    def _build_dense_screen_matrix(self, info) -> DenseTileMatrix:
        screen_matrix = DenseTileMatrix()
        for mario_tile in self._get_mario(info):
            screen_matrix.add(mario_tile)
        for sprite in self._get_sprites(info):
            screen_matrix.add(sprite)
        for extended_sprite in self._get_extended_sprites(info):
            screen_matrix.add(extended_sprite)
        for tile in self._get_layer1_tiles(info):
            screen_matrix.add(tile)
        return screen_matrix

    def _get_mario(self, info: dict) -> list[UngriddedTile]:
        mario_tiles = []
        mario_x = info.get("x", 0)
        mario_y = info.get("y", 0)
        grid_x, grid_y = self._coords_absolute_to_cam_grid(mario_x, mario_y, info)
        grid_y += 1 # Fixing shift in mario position
        grid_x += 1
        if not self._are_grid_coords_in_cam_view(grid_x, grid_y):
            # self.logger.warning("Mario out of bounds")
            return []
        mario_priority = info.get("mario_priority", 2)
        layer = 1 if mario_priority == 3 else 3
        mario_tiles.append(UngriddedTile(type=TileType.MARIO, id=1, layer=layer, x=grid_x, y=grid_y))
        is_big = info.get('mario_powerup', 0) > 0
        if is_big and grid_y > 0:
            mario_tiles.append(UngriddedTile(type=TileType.MARIO, id=2, layer=layer, x=grid_x, y=grid_y - 1))
        return mario_tiles

    def _get_sprites(self, info: dict) -> list[UngriddedTile]:
        sprites = []
        for sprite_index in range(12):
            sprite = self._get_sprite_at_index(sprite_index, info)
            if sprite is not None:
                sprites.append(sprite)
        return sprites

    def _get_sprite_at_index(self, index: int, info: dict) -> Optional[UngriddedTile]:
        if info.get(f"sprite_status_{index}", 0) == 0:
            return None
        sprite_x = self._combine_low_and_high_bits(info.get(f"sprite_x_low_{index}", 0), info.get(f"sprite_x_high_{index}", 0))
        sprite_y = self._combine_low_and_high_bits(info.get(f"sprite_y_low_{index}", 0), info.get(f"sprite_y_high_{index}", 0))
        sprite_type = info.get(f"sprite_type_{index}", 0)
        grid_x, grid_y = self._coords_absolute_to_cam_grid(sprite_x, sprite_y, info)
        grid_x += 1
        if not self._are_grid_coords_in_cam_view(grid_x, grid_y):
            # self.logger.warning(f"Sprite {index} out of bounds")
            return None
        priority_byte = info.get(f"sprite_priority_{index}", 0)
        sprite_priority = priority_byte & 0x03
        layer_map = {3: 1, 2: 3, 1: 5, 0: 6}
        layer = layer_map.get(sprite_priority, 3)
        return UngriddedTile(type=TileType.SPRITE, id=sprite_type, layer=layer, x=grid_x, y=grid_y)

    def _get_extended_sprites(self, info: dict) -> list[UngriddedTile]:
        extended_sprites = []
        for sprite_index in range(10):
            extended_sprite = self._get_extended_sprite_at_index(sprite_index, info)
            if extended_sprite is not None:
                extended_sprites.append(extended_sprite)
        return extended_sprites

    def _get_extended_sprite_at_index(self, index: int, info: dict) -> Optional[UngriddedTile]:
        sprite_type = info.get(f"ext_sprite_type_{index}", 0)
        if sprite_type == 0:
            return None
        sprite_x = self._combine_low_and_high_bits(info.get(f"ext_sprite_x_low_{index}", 0), info.get(f"ext_sprite_x_high_{index}", 0))
        sprite_y = self._combine_low_and_high_bits(info.get(f"ext_sprite_y_low_{index}", 0), info.get(f"ext_sprite_y_high_{index}", 0))
        grid_x, grid_y = self._coords_absolute_to_cam_grid(sprite_x, sprite_y, info)
        grid_x += 1
        if not self._are_grid_coords_in_cam_view(grid_x, grid_y):
            # self.logger.warning(f"Extended sprite {index} out of bounds")
            return None
        return UngriddedTile(type=TileType.EXTENDED_SPRITE, id=sprite_type+200, layer=3, x=grid_x, y=grid_y)

    def _get_layer1_tiles(self, info: dict) -> list[UngriddedTile]:
        ram = self.env.unwrapped.get_ram()
        RAM_BASE_OFFSET = 0xF000
        ATTR_BASE_OFFSET = RAM_BASE_OFFSET + 0x10000
        cam_x, cam_y = self._get_camera_coords(info)
        tiles = []

        # Iterate through the 14x16 view of the camera
        for row in range(self.SCREEN_ROWS):
            for col in range(self.SCREEN_COLUMNS):
                # Calculate world pixel coordinates
                world_x = cam_x + (col * 16)
                world_y = cam_y + (row * 16)

                # 1. Convert pixels to tile coordinates
                tx = world_x // 16
                ty = world_y // 16

                # 2. Safety check: SMW levels are 27 tiles high (0 to 26)
                if ty < 0 or ty >= 27:
                    continue

                # 3. Determine which screen (16-tile wide chunk) we are in
                screen_num = tx // 16

                # 4. Local X within that screen (0 to 15)
                lx = tx % 16

                # 5. Calculate the index based on the 27-row height (0x1B0 bytes per screen)
                # Index = (Screen * TilesPerScreen) + (Row * TilesPerRow) + Column
                tile_index = (screen_num * 432) + (ty * 16) + lx

                # 6. Safety check for the Map16 table limit (14336 bytes)
                if tile_index < 14336:
                    tile_id = ram[RAM_BASE_OFFSET + tile_index]
                    if tile_id != 0x25: # Sky tiles
                        attr_byte = ram[ATTR_BASE_OFFSET + tile_index]
                        has_priority = (attr_byte & 0x20) != 0
                        layer = 2 if has_priority else 4
                        tiles.append(UngriddedTile(
                            type=TileType.TERRAIN,
                            id=tile_id,
                            layer=layer,
                            x=col,
                            y=row
                        ))
        return tiles

    def _combine_low_and_high_bits(self, low: int, high: int) -> int:
        return (high << 8) | low

    def _are_grid_coords_in_cam_view(self, grid_x: int, grid_y: int) -> bool:
        return 0 <= grid_x < self.SCREEN_COLUMNS and 0 <= grid_y < self.SCREEN_ROWS

    def _coords_absolute_to_cam_grid(self, x: int, y: int, info: dict) -> tuple[int, int]:
        relative_x, relative_y = self._coords_absolute_to_cam_relative(x, y, info)
        grid_x, grid_y = relative_x // self.TILE_SIZE, relative_y // self.TILE_SIZE
        return grid_x, grid_y

    def _coords_absolute_to_cam_relative(self, x: int, y: int, info: dict) -> tuple[int, int]:
        cam_x, cam_y = self._get_camera_coords(info)
        relative_x = int(x) - cam_x
        relative_y = int(y) - cam_y
        return relative_x, relative_y

    def _get_camera_coords(self, info: dict):
        cam_x = info.get('cam_x', None)
        ram = None
        if cam_x is None:
            ram = self.env.unwrapped.get_ram()
            cam_x = self._combine_low_and_high_bits(ram[0x1A], ram[0x1B])
        cam_y = info.get('cam_y', None)
        if cam_y is None:
            if ram is None:
                ram = self.env.unwrapped.get_ram()
            cam_y = self._combine_low_and_high_bits(ram[0x1C], ram[0x1D])
        return int(cam_x), int(cam_y)

In [26]:
import cv2

class DebugVisualizer():
    def __init__(self, screen_rows: int = 14, screen_columns: int = 16, tile_size: int = 16, render_grid=False):
        self.screen_rows = screen_rows
        self.screen_columns = screen_columns
        self.tile_size = tile_size
        self.render_grid = render_grid
        self.GRID_COLOR = (80, 80, 80)
        self.img_width = self.screen_columns * self.tile_size
        self.img_height = self.screen_rows * self.tile_size

    def overlay(self, original_frame, observation):
        game_view = original_frame.copy()
        if self.render_grid:
            game_view = self._draw_grid(game_view)
        matrix_view = self._get_observation_image(observation)
        return np.hstack((game_view, matrix_view))

    def _get_observation_image(self, observation):
        matrix_img = np.zeros((self.img_height, self.img_width, 3), dtype=np.uint8)
        if observation is not None:
            matrix_img = self._populate_objects(matrix_img, observation)
        if self.render_grid:
            matrix_img = self._draw_grid(matrix_img)
        return matrix_img

    def _populate_objects(self, matrix_img, observation):
        for row in range(self.screen_rows):
            for col in range(self.screen_columns):
                tile = observation[row][col]
                self._draw_tile(matrix_img, col, row, tile)
        return matrix_img

    def _draw_tile(self, img, x: int, y: int, tile: Tile) -> None:
        if tile["type"] == TileType.EMPTY:
            return
        color = self._choose_tile_color(tile)
        x_start = int(x * self.tile_size)
        y_start = int(y * self.tile_size)
        x_end = x_start + self.tile_size - 1
        y_end = y_start + self.tile_size - 1
        cv2.rectangle(img, (x_start, y_start), (x_end, y_end), color, -1)

    # def _choose_tile_color(self, tile: Tile) -> tuple[int, int, int]:
    #     if tile["type"] == TileType.EMPTY:
    #         return (0, 0, 0)
    #     if tile["type"] == TileType.MARIO:
    #         return (0, 255, 0)
    #     if tile["type"] == TileType.SPRITE or tile["type"] == TileType.EXTENDED_SPRITE:
    #         id = int(tile["id"])
    #         return (int((id * 40) % 256), int((id * 80) % 256), 100)
    #     if tile["type"] == TileType.TERRAIN:
    #         id = int(tile["id"])
    #         return (int((id * 10) % 256), int((id * 20) % 256), int((id * 30) % 256))
    #     return (255, 0, 0)

    def _choose_tile_color(self, tile: Tile) -> tuple[int, int, int]:
        if tile["type"] == TileType.EMPTY:
            return (0, 0, 0)

        # Base settings to ensure visibility
        t_type = tile["type"]
        t_id = int(tile["id"])
        h, s, v = 0, 200, 255 # Default (Red)

        # Map Types to specific Hue ranges (OpenCV Hue is 0-179)
        if t_type == TileType.MARIO:
            h, s, v = 60, 255, 255   # Bright Green

        elif t_type == TileType.TERRAIN:
            # Blue/Cyan range: Shift hue slightly by ID to distinguish tiles
            h = 100 + (t_id % 20)
            s = 150 + (t_id % 100)  # Variety in "vividness"

        elif t_type == TileType.BLOCK:
            # Orange/Yellow range
            h = 20 + (t_id % 15)
            s = 180 + (t_id % 75)

        elif t_type == TileType.SPRITE:
            # Purple/Magenta range
            h = 140 + (t_id % 20)
            v = 200 + (t_id % 55)   # Variety in brightness

        elif t_type == TileType.EXTENDED_SPRITE:
            # Deep Reds (as requested)
            h = 0 + (t_id % 10)
            s = 255

        else:
            # Unknown types - High-contrast Lime/Cyan
            h, s, v = 85, 255, 255

        # Convert the single HSV pixel to BGR for OpenCV
        hsv_pixel = np.uint8([[[h, s, v]]])
        bgr_pixel = cv2.cvtColor(hsv_pixel, cv2.COLOR_HSV2RGB)[0][0]

        return (int(bgr_pixel[0]), int(bgr_pixel[1]), int(bgr_pixel[2]))

    def _draw_grid(self, img):
        h, w = img.shape[:2]
        for x in range(0, w + 1, self.tile_size):
            cv2.line(img, (x, 0), (x, h), self.GRID_COLOR, 1)
        for y in range(0, h + 1, self.tile_size):
            cv2.line(img, (0, y), (w, y), self.GRID_COLOR, 1)
        return img

In [27]:
from cv2.gapi import add
from typing import Any

import gymnasium as gym


class EmulatorWrapper(gym.Wrapper):
    def __init__(self, env, max_episode_length, render_debug=False, render_grid=False, logger: Optional[Logger] = None):
        self.logger: Logger = logger if logger is not None else DummyLogger()
        self.env = env
        self._max_episode_length = max_episode_length
        self._world_parser = WorldParser(self.env, logger=logger)
        self._debug_visualizer = DebugVisualizer(render_grid=render_grid)
        self.observation_space = gym.spaces.Box(
            low=0, high=65535, shape=(14, 16), dtype=np.int32
        )
        self._current_step = 0
        self.render_debug = render_debug
        self.render_grid = render_grid
        self.observation = None
        self.seen_ids = set()
        super().__init__(self.env)

    def reset(self, **kwargs):
        self._current_step = 0
        obs, info = self.env.reset(**kwargs)
        matrix_obs = self._world_parser.get_screen_matrix(info)
        self._has_found_ram_offset = False
        return matrix_obs, info

    def render(self):
        original_frame = self.env.render()
        if not self.render_debug:
            return original_frame
        debug_frame = self._debug_visualizer.overlay(original_frame, self.observation)
        return debug_frame

    def step(self, action: list[int]) -> tuple[Any, float, bool, bool, dict]:
        _, _, _, _, info = self.env.step(action)
        self._current_step += 1
        self.observation = self._world_parser.get_screen_matrix(info)
        observation = self._world_parser.get_screen_matrix_simplified(info)
        self._update_seen_ids(observation, info)
        reward = 0.0
        terminated = self._has_died(info)
        truncated = self._has_reached_max_steps()
        new_info = self._reformat_info(info)
        return observation, reward, terminated, truncated, new_info

    def _has_died(self, info) -> bool:
        return info["lives"] < 4

    def _has_reached_max_steps(self) -> bool:
        return self._current_step >= self._max_episode_length

    def _reformat_info(self, info) -> dict:
        return {}

    def _update_seen_ids(self, observation, info) -> None:
        has_found_new_tiles = False
        rows = len(observation)
        cols = len(observation[0])
        for row in range(rows):
            for col in range(cols):
                tile_id = observation[row, col]
                if tile_id not in self.seen_ids:
                    self.logger.info(f"New tile id found: {tile_id:03x}")
                    self.seen_ids.add(tile_id)
                    has_found_new_tiles = True
        if has_found_new_tiles:
            self._debug_print_simplified_tile_ids(observation)
        #     self._debug_print_tile_map(info)
        #     self._debug_print_tile_map_from_ram(info)
        #     if not self._has_found_ram_offset:
        #         self._has_found_ram_offset = True
        #         ram_offset = self.find_ram_offset(info)

    # def _debug_print_tile_map(self, info):
    #     GROUND_SCREEN_OFFSET = 256
    #     accessed_memory = TILE_LAYER_1_LOW + GROUND_SCREEN_OFFSET
    #     self.logger.debug(f"--- Layer 1 RAM Dump (0x{accessed_memory:04x}) ---")
    #     rows = 16
    #     cols = 16
    #     for y in range(rows):
    #         row_str = ""
    #         for x in range(cols):
    #             idx = y * cols + x
    #             val = info.get(f"tile_layer1_low_{GROUND_SCREEN_OFFSET + idx}", 0)
    #             row_str += f"{val:02x} "
    #         self.logger.debug(row_str)

    # def _debug_print_tile_map_from_ram(self, info):
    #     # TILE_LAYER_1_LOW = 0xC900
    #     TILE_LAYER_1_LOW = 0xf100
    #     self.logger.debug(f"--- Layer 1 RAM Dump [DIRECT ACCESS] (0x{TILE_LAYER_1_LOW:04x}) ---")
    #     rows = 16
    #     cols = 16
    #     ram = self.env.unwrapped.get_ram()
    #     for y in range(rows):
    #         row_str = ""
    #         for x in range(cols):
    #             idx = y * cols + x
    #             adddress = TILE_LAYER_1_LOW + idx
    #             val = ram[adddress]
    #             row_str += f"{val:02x} "
    #         self.logger.debug(row_str)

    # def find_ram_offset(self, info: dict):
    #     ram = self.env.unwrapped.get_ram()
    #     import sys
    #     self.logger.info(f"ram[0] type: {type(ram[0])}, size: {sys.getsizeof(ram[0])}, value: {ram[0]}. ram length: {len(ram)}")
    #     known_sequence = []
    #     for i in range(256):
    #         known_sequence.append(info.get(f"tile_layer1_low_{256 + i}", 0))

    #     self.logger.info(f"Searching RAM for known tile sequence: {known_sequence}")
    #     for i in range(len(ram) - 32):
    #         match = True
    #         for j in range(len(known_sequence)):
    #             if ram[i + j] != known_sequence[j]:
    #                 match = False
    #                 break
    #         if match:
    #             self.logger.info(f"Found tile sequence at offset 0x{i:04x}")
    #             block_start = i - 256
    #             self.logger.info(f"Block start: 0x{block_start:04x}")
    #             return i - 512

    #     self.logger.error("Could not find the tilemap pattern in the raw RAM array.")
    #     return None

    def _debug_print_simplified_tile_ids(self, observation):
        self.logger.debug("--- Tile IDs ---")
        rows = 14
        cols = 16
        for y in range(rows):
            row_str = ""
            for x in range(cols):
                val = observation[y, x]
                row_str += f"{val:03x} "
            self.logger.debug(row_str)

In [28]:
WALK_RIGHT_ACTION = [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]

In [31]:
RUN_NAME = "debugging_memory"
LEVEL = "DonutPlains1"
LEVEL = "YoshiIsland2"
MAX_STEPS = 1500
LOG_LEVEL = "INFO"

In [32]:
import time

import stable_retro as retro
from gymnasium.wrappers import RecordVideo


logger = get_logger(RUN_NAME, LOG_LEVEL)
env = retro.make(game='SuperMarioWorld-Snes-v0', state=LEVEL, render_mode="rgb_array")
try:
    env = EmulatorWrapper(env, MAX_STEPS, render_debug=True, render_grid=True, logger=logger)
    env = RecordVideo(env, video_folder="./", name_prefix=RUN_NAME, episode_trigger=lambda x: True)
    obs = env.reset()
    done = False
    while not done:
        env.render()
        obs, reward, terminated, truncated, info = env.step(WALK_RIGHT_ACTION)
        # logger.debug(obs)
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.close()

2026-04-19 02:16:44 [INFO] Session log for run debugging_memory with level [INFO] initialized at: debugging_memory_20260419_021644.log
2026-04-19 02:16:44 [INFO] New tile id found: 000
2026-04-19 02:16:44 [INFO] New tile id found: 34b
2026-04-19 02:16:44 [INFO] New tile id found: 34d
2026-04-19 02:16:44 [INFO] New tile id found: 34e
2026-04-19 02:16:44 [INFO] New tile id found: 34c
2026-04-19 02:16:44 [INFO] New tile id found: 354
2026-04-19 02:16:44 [INFO] New tile id found: 349
2026-04-19 02:16:44 [INFO] New tile id found: 35f
2026-04-19 02:16:44 [INFO] New tile id found: 363
2026-04-19 02:16:44 [INFO] New tile id found: 357
2026-04-19 02:16:44 [INFO] New tile id found: 352
2026-04-19 02:16:44 [INFO] New tile id found: 34a
2026-04-19 02:16:44 [INFO] New tile id found: 35d
2026-04-19 02:16:44 [INFO] New tile id found: 35a
2026-04-19 02:16:44 [INFO] New tile id found: 345
2026-04-19 02:16:44 [INFO] New tile id found: 350
2026-04-19 02:16:44 [INFO] New tile id found: 351
2026-04-19 02:1